In [2]:
import sys
import subprocess
import json
import hashlib
import math
import platform
from pathlib import Path

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas>=2.0",
        "matplotlib>=3.7",
        "jinja2>=3.1",
    ],
    check=True,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass


# =============================================================================
# Configuration
# =============================================================================

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")

DATA_DIR = ROOT / "data/extracted/webqsp"
E2E_DIR = ROOT / "outputs/end_to_end_test_v1"
EXT_DIR = ROOT / "outputs/external_baselines_v1"
CTRL_DIR = ROOT / "outputs/controller_training_v1"
ANS_DIR = ROOT / "outputs/answer_recognizer_v1"

OUT_DIR = ROOT / "outputs/operating_point_and_cost_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_DATA = DATA_DIR / "test_simple.json"

E2E_PER_QUESTION = (
    E2E_DIR / "end_to_end_per_question_v1.jsonl"
)

E2E_SUMMARY = (
    E2E_DIR / "end_to_end_summary_v1.json"
)

EXT_PER_QUESTION = (
    EXT_DIR / "external_baseline_per_question_v1.jsonl"
)

EXT_SUMMARY = (
    EXT_DIR / "external_baseline_summary_v1.json"
)

EXT_MANIFEST = (
    EXT_DIR / "external_baseline_run_manifest_v1.json"
)

CONTROLLER_THRESHOLD = (
    CTRL_DIR / "dev_stop_threshold_v1.json"
)

ANSWER_THRESHOLD = (
    ANS_DIR / "answer_recognizer_threshold_v1.json"
)

REQUIRED_INPUTS = [
    TEST_DATA,
    E2E_PER_QUESTION,
    E2E_SUMMARY,
    EXT_PER_QUESTION,
    EXT_SUMMARY,
    EXT_MANIFEST,
    CONTROLLER_THRESHOLD,
    ANSWER_THRESHOLD,
]

missing_files = [
    str(path)
    for path in REQUIRED_INPUTS
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing_files)
    )

BOOTSTRAP_SAMPLES = 5000
RANDOM_SEED = 20260923

print("=" * 108)
print(
    "SETSTOP-KGQA OPERATING-POINT "
    "AND COST-AWARE EXTERNAL COMPARISON"
)
print("=" * 108)
print("Runtime: CPU — GPU is not needed")
print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("Bootstrap samples:", BOOTSTRAP_SAMPLES)
print("Output directory:", OUT_DIR)
print("=" * 108)


# =============================================================================
# Helper functions
# =============================================================================

def load_json(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path):
    rows = []

    with open(path, encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path}, "
                    f"line {line_number}"
                ) from exc

    return rows


def convert_json_value(value):
    if isinstance(value, dict):
        return {
            str(key): convert_json_value(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            convert_json_value(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if not np.isfinite(value):
            return None
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if (
        isinstance(value, float)
        and not math.isfinite(value)
    ):
        return None

    return value


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(
            convert_json_value(obj),
            handle,
            indent=2,
            ensure_ascii=False,
        )


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def pick(row, *keys, default=None):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]

    return default


def safe_float(value, default=np.nan):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(default)


METHOD_ALIASES = {
    "relation_fixed_e2e": "Relation fixed",
    "relation_fixed": "Relation fixed",
    "relation": "Relation fixed",

    "model_fixed_e2e": "Learned ranker, fixed budget",
    "model_fixed": "Learned ranker, fixed budget",
    "learned_ranker_fixed": "Learned ranker, fixed budget",

    "setstop_e2e": "SetStop",
    "setstop": "SetStop",

    "oracle_rank_e2e": "Oracle ranker",
    "oracle_rank": "Oracle ranker",
    "oracle": "Oracle ranker",

    "nsm+h": "NSM+h",
    "nsm_h": "NSM+h",

    "rearev": "ReaRev",
}


def canonical_method(value):
    raw = str(value).strip()
    return METHOD_ALIASES.get(
        raw.lower(),
        raw,
    )


def normalize_record(row, source):
    """
    Normalize both internal SetStop records and external
    NSM+h/ReaRev records.

    Important schema difference:
      Internal: true_positive_count
      External: true_positives
    """

    qid = pick(
        row,
        "qid",
        "id",
        "question_id",
    )

    method_raw = pick(
        row,
        "policy",
        "method",
        "model",
        "policy_name",
    )

    if qid is None or method_raw is None:
        raise KeyError(
            f"Cannot identify qid/method in {source} row. "
            f"Available keys: {sorted(row.keys())}"
        )

    # Corrected aliases for both schemas.
    tp_value = pick(
        row,
        "true_positive_count",
        "true_positives",
        "tp",
    )

    fp_value = pick(
        row,
        "false_positive_count",
        "false_positives",
        "fp",
    )

    fn_value = pick(
        row,
        "false_negative_count",
        "false_negatives",
        "fn",
    )

    if (
        tp_value is None
        or fp_value is None
        or fn_value is None
    ):
        raise KeyError(
            f"Missing TP/FP/FN counts for {source} "
            f"row qid={qid}. "
            f"Available keys: {sorted(row.keys())}"
        )

    tp = int(tp_value)
    fp = int(fp_value)
    fn = int(fn_value)

    default_precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    default_recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 1.0
    )

    default_f1 = (
        2
        * default_precision
        * default_recall
        / (
            default_precision
            + default_recall
        )
        if default_precision + default_recall > 0
        else 0.0
    )

    precision = safe_float(
        pick(
            row,
            "precision",
            "answer_precision",
        ),
        default_precision,
    )

    recall = safe_float(
        pick(
            row,
            "recall",
            "answer_recall",
        ),
        default_recall,
    )

    f1 = safe_float(
        pick(
            row,
            "f1",
            "answer_f1",
        ),
        default_f1,
    )

    exact_match = safe_float(
        pick(
            row,
            "exact_match",
            "exact",
        ),
        0.0,
    )

    complete_recall = safe_float(
        pick(
            row,
            "complete_recall",
            "complete",
        ),
        0.0,
    )

    gold_answer_count = int(
        pick(
            row,
            "gold_answer_count",
            default=tp + fn,
        )
    )

    predicted_answer_count = int(
        pick(
            row,
            "predicted_answer_count",
            default=tp + fp,
        )
    )

    return {
        "qid": str(qid),
        "method": canonical_method(method_raw),
        "source": source,

        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": exact_match,
        "complete_recall": complete_recall,

        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,

        "gold_answer_count": gold_answer_count,
        "predicted_answer_count": predicted_answer_count,

        "actions": safe_float(
            pick(
                row,
                "actions",
                "number_of_actions",
            )
        ),

        "retrieved_edges": safe_float(
            pick(
                row,
                "retrieved_edges",
                "edges",
            )
        ),

        "premature_stop": safe_float(
            pick(
                row,
                "premature_stop",
            )
        ),

        "over_search_actions": safe_float(
            pick(
                row,
                "over_search_actions",
            )
        ),

        "frontier_truncations": safe_float(
            pick(
                row,
                "frontier_truncations",
            )
        ),

        "stop_reason": pick(
            row,
            "stop_reason",
            default=None,
        ),
    }


def graph_size(subgraph):
    entities = subgraph.get(
        "entities",
        [],
    )

    triples = None

    for key in (
        "tuples",
        "triples",
        "edges",
    ):
        if key in subgraph:
            triples = subgraph[key]
            break

    if triples is None:
        raise KeyError(
            "No tuples/triples/edges key in subgraph. "
            f"Available keys: {sorted(subgraph.keys())}"
        )

    return len(entities), len(triples)


def mean_ci(values, bootstrap_indices):
    values = np.asarray(
        values,
        dtype=float,
    )

    if not np.all(np.isfinite(values)):
        raise ValueError(
            "Non-finite values found while "
            "computing confidence interval"
        )

    estimate = float(
        np.mean(values)
    )

    bootstrap_draws = values[
        bootstrap_indices
    ].mean(axis=1)

    lower, upper = np.quantile(
        bootstrap_draws,
        [0.025, 0.975],
    )

    return {
        "mean": estimate,
        "ci95_lower": float(lower),
        "ci95_upper": float(upper),
        "bootstrap_draws": bootstrap_draws,
    }


def paired_difference(
    reference,
    comparator,
    bootstrap_indices,
    positive_means,
):
    reference = np.asarray(
        reference,
        dtype=float,
    )

    comparator = np.asarray(
        comparator,
        dtype=float,
    )

    difference = (
        reference
        - comparator
    )

    if not np.all(np.isfinite(difference)):
        raise ValueError(
            "Non-finite paired differences encountered"
        )

    bootstrap_draws = difference[
        bootstrap_indices
    ].mean(axis=1)

    lower, upper = np.quantile(
        bootstrap_draws,
        [0.025, 0.975],
    )

    return {
        "mean_difference": float(
            np.mean(difference)
        ),

        "ci95_lower": float(lower),
        "ci95_upper": float(upper),

        "probability_positive": float(
            np.mean(bootstrap_draws > 0)
        ),

        "positive_means": positive_means,
    }


# =============================================================================
# 1. Verify frozen thresholds
# =============================================================================

controller_threshold_info = load_json(
    CONTROLLER_THRESHOLD
)

answer_threshold_info = load_json(
    ANSWER_THRESHOLD
)

external_manifest = load_json(
    EXT_MANIFEST
)

controller_threshold = float(
    controller_threshold_info[
        "selected_threshold"
    ]
)

answer_threshold = float(
    answer_threshold_info[
        "selected_threshold"
    ]
)

controller_selection_split = str(
    controller_threshold_info.get(
        "selection_split",
        "",
    )
).lower()

answer_selection_split = str(
    answer_threshold_info.get(
        "selection_split",
        "",
    )
).lower()

assert controller_selection_split.startswith(
    "dev"
), (
    "Controller threshold was not selected "
    "on development data"
)

assert answer_selection_split.startswith(
    "dev"
), (
    "Answer threshold was not selected "
    "on development data"
)

assert (
    external_manifest.get(
        "test_used_for_threshold_selection"
    )
    is False
), (
    "External test results were used "
    "for threshold selection"
)

threshold_rows = (
    controller_threshold_info.get(
        "threshold_results",
        [],
    )
)

if not threshold_rows:
    raise KeyError(
        "No threshold_results found in "
        f"{CONTROLLER_THRESHOLD}"
    )

threshold_df = (
    pd.DataFrame(threshold_rows)
    .sort_values("threshold")
    .reset_index(drop=True)
)

threshold_csv = (
    OUT_DIR
    / "dev_stop_threshold_sweep_v1.csv"
)

threshold_df.to_csv(
    threshold_csv,
    index=False,
)

frozen_operating_points = {
    "selection_policy": (
        "All learned cutoffs were selected on "
        "development data. No test-set retuning."
    ),

    "controller_threshold": (
        controller_threshold
    ),

    "answer_recognizer_threshold": (
        answer_threshold
    ),

    "maximum_actions": 12,
    "maximum_frontier_candidates": 16,

    "operating_points": {
        "efficiency": {
            "name": "SetStop",
            "learned_ranking": True,
            "learned_stopping": True,
            "controller_threshold": (
                controller_threshold
            ),
        },

        "fixed_budget_reference": {
            "name": (
                "Learned ranker, fixed budget"
            ),
            "learned_ranking": True,
            "learned_stopping": False,
            "maximum_actions": 12,
        },

        "oracle_diagnostic": {
            "name": "Oracle ranker",
            "uses_gold_gain": True,
            "eligible_as_real_baseline": False,
        },
    },
}

frozen_path = (
    OUT_DIR
    / "frozen_operating_points_v1.json"
)

save_json(
    frozen_operating_points,
    frozen_path,
)

fig, ax = plt.subplots(
    figsize=(7.4, 4.6)
)

ax.plot(
    threshold_df["threshold"],
    threshold_df["stop_f1"],
    linewidth=2.2,
    label="STOP F1",
)

ax.plot(
    threshold_df["threshold"],
    threshold_df["premature_stop_rate"],
    linewidth=1.8,
    label="Premature-stop rate",
)

ax.plot(
    threshold_df["threshold"],
    threshold_df["over_search_rate"],
    linewidth=1.8,
    label="Over-search rate",
)

ax.axvline(
    controller_threshold,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=(
        f"Frozen threshold = "
        f"{controller_threshold:.2f}"
    ),
)

ax.set_xlabel(
    "Stopping threshold "
    "(development set only)"
)

ax.set_ylabel("Rate")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.25)

ax.legend(
    frameon=False,
    fontsize=9,
)

fig.tight_layout()

DEV_CURVE_PDF = (
    OUT_DIR
    / "dev_stop_threshold_curve_v1.pdf"
)

DEV_CURVE_PNG = (
    OUT_DIR
    / "dev_stop_threshold_curve_v1.png"
)

fig.savefig(
    DEV_CURVE_PDF,
    bbox_inches="tight",
)

fig.savefig(
    DEV_CURVE_PNG,
    dpi=220,
    bbox_inches="tight",
)

plt.close(fig)

print("\nFrozen operating point")
print(
    "Controller threshold:",
    controller_threshold,
)
print(
    "Answer threshold:",
    answer_threshold,
)
print(
    "No threshold is changed using test results."
)


# =============================================================================
# 2. Load internal and external per-question records
# =============================================================================

internal_raw = load_jsonl(
    E2E_PER_QUESTION
)

external_raw = load_jsonl(
    EXT_PER_QUESTION
)

print(
    "\nInternal policy-question rows:",
    len(internal_raw),
)

print(
    "External model-question rows:",
    len(external_raw),
)

internal_records = [
    normalize_record(
        row,
        source="internal",
    )
    for row in internal_raw
]

external_records = [
    normalize_record(
        row,
        source="external",
    )
    for row in external_raw
]

all_records = (
    internal_records
    + external_records
)

required_methods = [
    "Relation fixed",
    "Learned ranker, fixed budget",
    "SetStop",
    "Oracle ranker",
    "NSM+h",
    "ReaRev",
]

available_methods = sorted({
    row["method"]
    for row in all_records
})

print(
    "\nAvailable normalized methods:"
)

for method in available_methods:
    count = sum(
        1
        for row in all_records
        if row["method"] == method
    )

    print(
        f"  {method}: {count}"
    )

missing_methods = [
    method
    for method in required_methods
    if method not in available_methods
]

if missing_methods:
    raise AssertionError(
        f"Missing methods: {missing_methods}. "
        f"Available methods: {available_methods}"
    )

controlled_records = [
    row
    for row in all_records
    if row["method"] in required_methods
]

results = pd.DataFrame(
    controlled_records
)

duplicates = results.duplicated(
    ["method", "qid"],
    keep=False,
)

if duplicates.any():
    examples = (
        results.loc[
            duplicates,
            ["method", "qid"],
        ]
        .head(10)
        .to_dict("records")
    )

    raise AssertionError(
        "Duplicate method-question records: "
        f"{examples}"
    )


# =============================================================================
# 3. Verify identical 736-question test subset
# =============================================================================

qid_sets = {
    method: set(
        results.loc[
            results["method"] == method,
            "qid",
        ]
    )
    for method in required_methods
}

reference_qids = qid_sets[
    "SetStop"
]

for method, qids in qid_sets.items():
    if qids != reference_qids:
        raise AssertionError(
            f"QID mismatch for {method}: "
            f"{len(qids)} vs SetStop "
            f"{len(reference_qids)}; "
            f"missing="
            f"{len(reference_qids - qids)}, "
            f"extra="
            f"{len(qids - reference_qids)}"
        )

assert len(reference_qids) == 736, (
    "Expected 736 controlled test questions, "
    f"but found {len(reference_qids)}"
)

computed_qid_hash = hashlib.sha256(
    "\n".join(
        sorted(reference_qids)
    ).encode("utf-8")
).hexdigest()

expected_qid_hash = (
    external_manifest.get(
        "test_qid_sha256"
    )
)

if expected_qid_hash:
    assert (
        computed_qid_hash
        == expected_qid_hash
    ), (
        "Controlled test-QID hash mismatch"
    )

print(
    "\nControlled test subset verified:",
    len(reference_qids),
    "questions",
)

print(
    "QID SHA-256:",
    computed_qid_hash,
)


# =============================================================================
# 4. Load complete graph size for each test question
# =============================================================================

graph_information = {}

with open(
    TEST_DATA,
    encoding="utf-8",
) as handle:
    for line in handle:
        if not line.strip():
            continue

        row = json.loads(line)
        qid = str(row["id"])

        if qid not in reference_qids:
            continue

        node_count, edge_count = graph_size(
            row["subgraph"]
        )

        graph_information[qid] = {
            "full_graph_nodes": node_count,
            "full_graph_edges": edge_count,
        }

if set(graph_information) != reference_qids:
    missing_graphs = (
        reference_qids
        - set(graph_information)
    )

    raise AssertionError(
        "Could not find graph information for "
        f"{len(missing_graphs)} questions"
    )

results["full_graph_nodes"] = (
    results["qid"].map(
        lambda qid: (
            graph_information[qid][
                "full_graph_nodes"
            ]
        )
    )
)

results["full_graph_edges"] = (
    results["qid"].map(
        lambda qid: (
            graph_information[qid][
                "full_graph_edges"
            ]
        )
    )
)

external_mask = (
    results["source"] == "external"
)

missing_internal_edges = results.loc[
    (~external_mask)
    & results["retrieved_edges"].isna(),
    ["method", "qid"],
]

if not missing_internal_edges.empty:
    raise AssertionError(
        "Internal retrieved-edge values are missing:\n"
        + missing_internal_edges
        .head(10)
        .to_string(index=False)
    )

# Internal models: actual retrieved edges.
# External GNNs: all input graph edges are exposed.
# This is a structural-access proxy, not runtime or FLOPs.

results["input_edges_exposed"] = (
    results["retrieved_edges"]
)

results.loc[
    external_mask,
    "input_edges_exposed",
] = results.loc[
    external_mask,
    "full_graph_edges",
]

results["edge_exposure_fraction"] = (
    results["input_edges_exposed"]
    / results["full_graph_edges"]
)

results["cost_definition"] = np.where(
    external_mask,
    "all input-subgraph triples exposed",
    "triples retrieved by frontier actions",
)


# =============================================================================
# 5. Build aligned method frames
# =============================================================================

ordered_qids = sorted(
    reference_qids
)

number_of_questions = len(
    ordered_qids
)

method_frames = {}

for method in required_methods:
    frame = (
        results[
            results["method"] == method
        ]
        .set_index("qid")
        .loc[ordered_qids]
        .copy()
    )

    required_numeric_columns = [
        "precision",
        "recall",
        "f1",
        "exact_match",
        "complete_recall",
        "true_positives",
        "false_positives",
        "false_negatives",
        "input_edges_exposed",
    ]

    for column in required_numeric_columns:
        values = frame[
            column
        ].to_numpy(dtype=float)

        if not np.all(
            np.isfinite(values)
        ):
            raise ValueError(
                f"Non-finite values for {method}, "
                f"column={column}"
            )

    method_frames[method] = frame

rng = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_indices = rng.integers(
    0,
    number_of_questions,
    size=(
        BOOTSTRAP_SAMPLES,
        number_of_questions,
    ),
)


# =============================================================================
# 6. Calculate aggregate metrics
# =============================================================================

summary_rows = []

for method in required_methods:
    frame = method_frames[method]

    f1_statistics = mean_ci(
        frame["f1"],
        bootstrap_indices,
    )

    exact_statistics = mean_ci(
        frame["exact_match"],
        bootstrap_indices,
    )

    complete_statistics = mean_ci(
        frame["complete_recall"],
        bootstrap_indices,
    )

    edge_statistics = mean_ci(
        frame["input_edges_exposed"],
        bootstrap_indices,
    )

    total_tp = int(
        frame["true_positives"].sum()
    )

    total_fp = int(
        frame["false_positives"].sum()
    )

    total_fn = int(
        frame["false_negatives"].sum()
    )

    micro_precision = (
        total_tp
        / (total_tp + total_fp)
        if total_tp + total_fp > 0
        else 0.0
    )

    micro_recall = (
        total_tp
        / (total_tp + total_fn)
        if total_tp + total_fn > 0
        else 0.0
    )

    micro_f1 = (
        2
        * micro_precision
        * micro_recall
        / (
            micro_precision
            + micro_recall
        )
        if micro_precision + micro_recall > 0
        else 0.0
    )

    if frame["actions"].notna().any():
        average_actions = float(
            frame["actions"].mean()
        )
    else:
        average_actions = np.nan

    average_exposure_fraction = float(
        frame[
            "edge_exposure_fraction"
        ].mean()
    )

    summary_rows.append({
        "Method": method,
        "Questions": number_of_questions,

        "Macro Precision": float(
            frame["precision"].mean()
        ),

        "Macro Recall": float(
            frame["recall"].mean()
        ),

        "Macro F1": (
            f1_statistics["mean"]
        ),

        "Macro F1 CI95 Low": (
            f1_statistics["ci95_lower"]
        ),

        "Macro F1 CI95 High": (
            f1_statistics["ci95_upper"]
        ),

        "Micro Precision": (
            micro_precision
        ),

        "Micro Recall": (
            micro_recall
        ),

        "Micro F1": micro_f1,

        "Exact Match": (
            exact_statistics["mean"]
        ),

        "Exact CI95 Low": (
            exact_statistics["ci95_lower"]
        ),

        "Exact CI95 High": (
            exact_statistics["ci95_upper"]
        ),

        "Complete Recall": (
            complete_statistics["mean"]
        ),

        "Complete CI95 Low": (
            complete_statistics[
                "ci95_lower"
            ]
        ),

        "Complete CI95 High": (
            complete_statistics[
                "ci95_upper"
            ]
        ),

        "Mean Actions": (
            average_actions
        ),

        "Mean Input Edges Exposed": (
            edge_statistics["mean"]
        ),

        "Edges CI95 Low": (
            edge_statistics["ci95_lower"]
        ),

        "Edges CI95 High": (
            edge_statistics["ci95_upper"]
        ),

        "Mean Fraction of Input Graph Exposed": (
            average_exposure_fraction
        ),

        "Macro F1 per 100 Exposed Edges": (
            100.0
            * f1_statistics["mean"]
            / edge_statistics["mean"]
            if edge_statistics["mean"] > 0
            else np.nan
        ),

        "Total True Positives": total_tp,
        "Total False Positives": total_fp,
        "Total False Negatives": total_fn,

        "Cost Is Structural Proxy": True,
    })

summary_df = pd.DataFrame(
    summary_rows
)

method_order = {
    method: index
    for index, method
    in enumerate(required_methods)
}

summary_df["_order"] = (
    summary_df["Method"].map(
        method_order
    )
)

summary_df = (
    summary_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


# =============================================================================
# 7. Determine the non-oracle Pareto frontier
# =============================================================================

non_oracle_table = summary_df[
    summary_df["Method"] != "Oracle ranker"
].copy()

pareto_methods = []

for _, candidate in non_oracle_table.iterrows():
    dominated = False

    for _, other in non_oracle_table.iterrows():
        if (
            candidate["Method"]
            == other["Method"]
        ):
            continue

        no_worse = (
            other["Macro F1"]
            >= candidate["Macro F1"]
            and
            other["Mean Input Edges Exposed"]
            <= candidate[
                "Mean Input Edges Exposed"
            ]
        )

        strictly_better = (
            other["Macro F1"]
            > candidate["Macro F1"]
            or
            other["Mean Input Edges Exposed"]
            < candidate[
                "Mean Input Edges Exposed"
            ]
        )

        if no_worse and strictly_better:
            dominated = True
            break

    if not dominated:
        pareto_methods.append(
            candidate["Method"]
        )

summary_df[
    "Non-Oracle Pareto Frontier"
] = summary_df["Method"].isin(
    pareto_methods
)


# =============================================================================
# 8. Save main result tables
# =============================================================================

COMPARISON_CSV = (
    OUT_DIR
    / "cost_aware_external_comparison_v1.csv"
)

summary_df.to_csv(
    COMPARISON_CSV,
    index=False,
)

latex_columns = [
    "Method",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Micro F1",
    "Exact Match",
    "Complete Recall",
    "Mean Actions",
    "Mean Input Edges Exposed",
]

latex_table = (
    summary_df[latex_columns]
    .to_latex(
        index=False,
        float_format=lambda value: (
            "--"
            if pd.isna(value)
            else f"{value:.4f}"
        ),
        escape=True,
    )
)

LATEX_TABLE = (
    OUT_DIR
    / "cost_aware_external_comparison_v1.tex"
)

with open(
    LATEX_TABLE,
    "w",
    encoding="utf-8",
) as handle:
    handle.write(latex_table)

PER_QUESTION_OUTPUT = (
    OUT_DIR
    / "cost_aware_per_question_v1.jsonl"
)

with open(
    PER_QUESTION_OUTPUT,
    "w",
    encoding="utf-8",
) as handle:
    output_rows = (
        results
        .sort_values(
            ["method", "qid"]
        )
        .to_dict("records")
    )

    for row in output_rows:
        row = convert_json_value(row)

        handle.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + "\n"
        )

display_columns = [
    "Method",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Micro F1",
    "Exact Match",
    "Complete Recall",
    "Mean Actions",
    "Mean Input Edges Exposed",
    "Mean Fraction of Input Graph Exposed",
    "Non-Oracle Pareto Frontier",
]

print("\nControlled main comparison")

print(
    summary_df[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.4f}"
        ),
    )
)


# =============================================================================
# 9. Paired bootstrap comparison against SetStop
# =============================================================================

setstop_frame = method_frames[
    "SetStop"
]

paired_results = {
    "reference_method": "SetStop",

    "questions": number_of_questions,

    "bootstrap_samples": (
        BOOTSTRAP_SAMPLES
    ),

    "random_seed": RANDOM_SEED,

    "interpretation": {
        "accuracy_differences": (
            "Positive means SetStop is better."
        ),

        "input_edges_saved": (
            "Positive means SetStop exposes "
            "fewer input graph triples."
        ),

        "actions_saved": (
            "Positive means SetStop takes "
            "fewer explicit frontier actions."
        ),
    },

    "comparisons": {},
}

comparison_methods = [
    method
    for method in required_methods
    if method not in (
        "SetStop",
        "Oracle ranker",
    )
]

for comparator in comparison_methods:
    comparator_frame = method_frames[
        comparator
    ]

    comparison = {
        "macro_f1": paired_difference(
            setstop_frame[
                "f1"
            ].to_numpy(),

            comparator_frame[
                "f1"
            ].to_numpy(),

            bootstrap_indices,

            "SetStop has higher macro F1",
        ),

        "exact_match": paired_difference(
            setstop_frame[
                "exact_match"
            ].to_numpy(),

            comparator_frame[
                "exact_match"
            ].to_numpy(),

            bootstrap_indices,

            "SetStop has higher exact match",
        ),

        "complete_recall": paired_difference(
            setstop_frame[
                "complete_recall"
            ].to_numpy(),

            comparator_frame[
                "complete_recall"
            ].to_numpy(),

            bootstrap_indices,

            (
                "SetStop has higher "
                "complete recall"
            ),
        ),

        # Reverse the operands because positive means
        # that the comparator uses more edges.
        "input_edges_saved": paired_difference(
            comparator_frame[
                "input_edges_exposed"
            ].to_numpy(),

            setstop_frame[
                "input_edges_exposed"
            ].to_numpy(),

            bootstrap_indices,

            (
                "SetStop exposes fewer "
                "input triples"
            ),
        ),
    }

    if comparator_frame[
        "actions"
    ].notna().all():

        comparison[
            "actions_saved"
        ] = paired_difference(
            comparator_frame[
                "actions"
            ].to_numpy(),

            setstop_frame[
                "actions"
            ].to_numpy(),

            bootstrap_indices,

            (
                "SetStop takes fewer "
                "explicit actions"
            ),
        )

    else:
        comparison[
            "actions_saved"
        ] = None

    comparator_mean_edges = float(
        comparator_frame[
            "input_edges_exposed"
        ].mean()
    )

    setstop_mean_edges = float(
        setstop_frame[
            "input_edges_exposed"
        ].mean()
    )

    comparison[
        "mean_edge_reduction_percent"
    ] = float(
        100.0
        * (
            1.0
            - setstop_mean_edges
            / comparator_mean_edges
        )
    )

    paired_results[
        "comparisons"
    ][comparator] = comparison

PAIRED_OUTPUT = (
    OUT_DIR
    / "paired_bootstrap_external_v1.json"
)

save_json(
    paired_results,
    PAIRED_OUTPUT,
)

print(
    "\nPaired comparisons: "
    "SetStop minus comparator"
)

for comparator, comparison in (
    paired_results[
        "comparisons"
    ].items()
):
    f1_result = comparison[
        "macro_f1"
    ]

    edge_result = comparison[
        "input_edges_saved"
    ]

    print(
        f"{comparator:30s} "
        f"Delta F1="
        f"{f1_result['mean_difference']:+.4f} "
        f"["
        f"{f1_result['ci95_lower']:+.4f}, "
        f"{f1_result['ci95_upper']:+.4f}"
        f"] | "
        f"edges saved="
        f"{edge_result['mean_difference']:+.2f} "
        f"("
        f"{comparison['mean_edge_reduction_percent']:+.2f}"
        f"%)"
    )


# =============================================================================
# 10. Publication-ready accuracy/cost figure
# =============================================================================

plot_df = summary_df[
    summary_df["Method"] != "Oracle ranker"
].copy()

colors = {
    "Relation fixed": "#9AA0A6",
    "Learned ranker, fixed budget": "#4285F4",
    "SetStop": "#EA4335",
    "NSM+h": "#34A853",
    "ReaRev": "#7E57C2",
}

markers = {
    "Relation fixed": "o",
    "Learned ranker, fixed budget": "s",
    "SetStop": "*",
    "NSM+h": "D",
    "ReaRev": "P",
}

fig, ax = plt.subplots(
    figsize=(8.4, 5.5)
)

for _, row in plot_df.iterrows():
    method = row["Method"]

    lower_error = (
        row["Macro F1"]
        - row["Macro F1 CI95 Low"]
    )

    upper_error = (
        row["Macro F1 CI95 High"]
        - row["Macro F1"]
    )

    y_error = np.array([
        [lower_error],
        [upper_error],
    ])

    ax.errorbar(
        row["Mean Input Edges Exposed"],
        row["Macro F1"],

        yerr=y_error,

        fmt=markers[method],

        markersize=(
            14
            if method == "SetStop"
            else 8
        ),

        color=colors[method],
        ecolor=colors[method],

        elinewidth=1.3,
        capsize=3,

        label=method,
    )

ax.set_xscale("log")

ax.set_xlabel(
    "Mean input triples exposed "
    "(log scale; structural proxy, not FLOPs)"
)

ax.set_ylabel(
    "Macro answer-set F1"
)

ax.set_title(
    "Accuracy–graph-access trade-off on "
    "the 736-question WebQSP subset"
)

ax.grid(
    alpha=0.25,
    which="both",
)

ax.legend(
    frameon=False,
    fontsize=9,
    loc="best",
)

fig.tight_layout()

PARETO_PDF = (
    OUT_DIR
    / "accuracy_cost_pareto_v1.pdf"
)

PARETO_PNG = (
    OUT_DIR
    / "accuracy_cost_pareto_v1.png"
)

fig.savefig(
    PARETO_PDF,
    bbox_inches="tight",
)

fig.savefig(
    PARETO_PNG,
    dpi=240,
    bbox_inches="tight",
)

plt.close(fig)


# =============================================================================
# 11. Generate final paper-oriented summary
# =============================================================================

setstop_summary = summary_df[
    summary_df["Method"] == "SetStop"
].iloc[0]

fixed_summary = summary_df[
    summary_df["Method"]
    == "Learned ranker, fixed budget"
].iloc[0]

nsm_summary = summary_df[
    summary_df["Method"] == "NSM+h"
].iloc[0]

rearev_summary = summary_df[
    summary_df["Method"] == "ReaRev"
].iloc[0]

final_summary = {
    "notebook": (
        "08_SetStop_Operating_Point_"
        "and_Cost_Analysis.ipynb"
    ),

    "runtime": "CPU",

    "dataset": (
        "WebQSP controlled multi-answer subset"
    ),

    "test_questions": (
        number_of_questions
    ),

    "thresholds_frozen_on_development": True,

    "controller_threshold": (
        controller_threshold
    ),

    "answer_recognizer_threshold": (
        answer_threshold
    ),

    "test_qid_sha256": (
        computed_qid_hash
    ),

    "setstop": {
        "macro_precision": float(
            setstop_summary[
                "Macro Precision"
            ]
        ),

        "macro_recall": float(
            setstop_summary[
                "Macro Recall"
            ]
        ),

        "macro_f1": float(
            setstop_summary[
                "Macro F1"
            ]
        ),

        "micro_f1": float(
            setstop_summary[
                "Micro F1"
            ]
        ),

        "exact_match": float(
            setstop_summary[
                "Exact Match"
            ]
        ),

        "complete_recall": float(
            setstop_summary[
                "Complete Recall"
            ]
        ),

        "mean_actions": float(
            setstop_summary[
                "Mean Actions"
            ]
        ),

        "mean_input_edges_exposed": float(
            setstop_summary[
                "Mean Input Edges Exposed"
            ]
        ),

        "mean_graph_exposure_fraction": float(
            setstop_summary[
                "Mean Fraction of Input Graph Exposed"
            ]
        ),
    },

    "accuracy_references": {
        "NSM+h_macro_f1": float(
            nsm_summary[
                "Macro F1"
            ]
        ),

        "ReaRev_macro_f1": float(
            rearev_summary[
                "Macro F1"
            ]
        ),
    },

    "setstop_vs_fixed_ranker": {
        "macro_f1_difference": float(
            setstop_summary["Macro F1"]
            - fixed_summary["Macro F1"]
        ),

        "exact_match_difference": float(
            setstop_summary["Exact Match"]
            - fixed_summary["Exact Match"]
        ),

        "complete_recall_difference": float(
            setstop_summary["Complete Recall"]
            - fixed_summary["Complete Recall"]
        ),

        "action_reduction_percent": float(
            100.0
            * (
                1.0
                - setstop_summary[
                    "Mean Actions"
                ]
                / fixed_summary[
                    "Mean Actions"
                ]
            )
        ),

        "edge_reduction_percent": float(
            100.0
            * (
                1.0
                - setstop_summary[
                    "Mean Input Edges Exposed"
                ]
                / fixed_summary[
                    "Mean Input Edges Exposed"
                ]
            )
        ),
    },

    "non_oracle_pareto_methods": (
        pareto_methods
    ),

    "cost_definition": (
        "For explicit-search methods, cost is "
        "the number of graph triples retrieved. "
        "For NSM+h and ReaRev, cost is the number "
        "of triples in the complete input subgraph. "
        "This measures graph exposure, not FLOPs, "
        "latency or energy."
    ),

    "paper_safe_conclusions": [
        (
            "SetStop improves macro answer-set F1 "
            "and exact match over its learned "
            "fixed-budget ranker while using fewer "
            "actions and graph triples."
        ),

        (
            "NSM+h and ReaRev remain substantially "
            "more accurate, so SetStop should be "
            "presented as an accuracy-efficiency "
            "trade-off rather than as a "
            "state-of-the-art accuracy system."
        ),

        (
            "All methods are compared using the "
            "same 736 question IDs and the same "
            "question-specific WebQSP graphs."
        ),

        (
            "All learned thresholds were frozen "
            "using development data before test "
            "evaluation."
        ),
    ],

    "limitations": [
        (
            "Input triples exposed is a structural "
            "graph-access proxy and is not directly "
            "equivalent to wall-clock runtime, "
            "memory use or neural FLOPs."
        ),

        (
            "NSM+h and ReaRev use full-graph "
            "message passing, whereas SetStop uses "
            "explicit sequential graph retrieval."
        ),

        (
            "The controlled main evaluation "
            "currently covers one dataset."
        ),
    ],
}

SUMMARY_OUTPUT = (
    OUT_DIR
    / "operating_point_and_cost_summary_v1.json"
)

save_json(
    final_summary,
    SUMMARY_OUTPUT,
)


# =============================================================================
# 12. Artifact integrity inventory
# =============================================================================

artifact_paths = [
    threshold_csv,
    frozen_path,
    DEV_CURVE_PDF,
    DEV_CURVE_PNG,
    COMPARISON_CSV,
    LATEX_TABLE,
    PER_QUESTION_OUTPUT,
    PAIRED_OUTPUT,
    PARETO_PDF,
    PARETO_PNG,
    SUMMARY_OUTPUT,
]

inventory = []

for path in artifact_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Expected output was not created: {path}"
        )

    inventory.append({
        "filename": path.name,
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

INVENTORY_OUTPUT = (
    OUT_DIR
    / "artifact_inventory_v1.json"
)

save_json(
    inventory,
    INVENTORY_OUTPUT,
)


# =============================================================================
# Completion report
# =============================================================================

print("\n" + "=" * 108)
print(
    "OPERATING-POINT AND COST ANALYSIS COMPLETED"
)
print("=" * 108)

completion_report = {
    "test_questions": (
        number_of_questions
    ),

    "controller_threshold": (
        controller_threshold
    ),

    "answer_threshold": (
        answer_threshold
    ),

    "setstop_macro_f1": float(
        setstop_summary[
            "Macro F1"
        ]
    ),

    "setstop_exact_match": float(
        setstop_summary[
            "Exact Match"
        ]
    ),

    "setstop_complete_recall": float(
        setstop_summary[
            "Complete Recall"
        ]
    ),

    "setstop_mean_actions": float(
        setstop_summary[
            "Mean Actions"
        ]
    ),

    "setstop_mean_edges_exposed": float(
        setstop_summary[
            "Mean Input Edges Exposed"
        ]
    ),

    "non_oracle_pareto_methods": (
        pareto_methods
    ),
}

print(
    json.dumps(
        completion_report,
        indent=2,
    )
)

print("\nSaved outputs:")

for item in inventory:
    print(
        f"{item['filename']:48s} "
        f"{item['size_bytes'] / 1024:.2f} KB  "
        f"{item['sha256'][:16]}"
    )

print(
    f"{'artifact_inventory_v1.json':48s} "
    f"{INVENTORY_OUTPUT.stat().st_size / 1024:.2f} KB"
)

print("\nSend these four files for analysis:")

print(SUMMARY_OUTPUT)
print(COMPARISON_CSV)
print(PAIRED_OUTPUT)
print(PARETO_PNG)

print("=" * 108)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SETSTOP-KGQA OPERATING-POINT AND COST-AWARE EXTERNAL COMPARISON
Runtime: CPU — GPU is not needed
Python: 3.13.15
Pandas: 2.2.3
Bootstrap samples: 5000
Output directory: /content/drive/MyDrive/SetStop_KGQA/outputs/operating_point_and_cost_v1

Frozen operating point
Controller threshold: 0.7100000000000001
Answer threshold: 0.6699999999999999
No threshold is changed using test results.

Internal policy-question rows: 5152
External model-question rows: 1472

Available normalized methods:
  Learned ranker, fixed budget: 736
  NSM+h: 736
  Oracle ranker: 736
  ReaRev: 736
  Relation fixed: 736
  SetStop: 736
  fifo_fixed_e2e: 736
  patience2_e2e: 736
  random_fixed_e2e: 736

Controlled test subset verified: 736 questions
QID SHA-256: a1881ab84e5ccafd18ae99f11df831202ec447b155056ea56cf16f9c9178ef85

Controlled main comparison
                      Method  Macro Pre